<a href="https://colab.research.google.com/github/psvprasad2003/SAMPLE_ML_MODELS/blob/main/madhav_executed_cp_gradient_boosted_tree_10a_build_sequence_failure_prediction_model_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10A - Gradient-Boosted Trees for Failure Prediction (Updated)

This production-ready notebook trains one histogram-based gradient-boosted binary classifier for each failure horizon: **7, 14, and 30 days**. It uses an engine-level failure-stratified split, training-only negative subsampling, balanced sample weights, validation PR-AUC model selection, natural validation/test prevalence, validation-derived alert thresholds, and permutation importance.


In [ ]:
# CELL 01 - Imports and production configuration
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance

ROOT = Path('/raid3/e296408/All_ECFRs_working')
BRANCH_A_DIR = ROOT / 'branch_a_ecfr_only'
PANEL_FILE = BRANCH_A_DIR / 'failure_prediction_panel' / 'ecfr_failure_prediction_panel.parquet'
OUTPUT_DIR = BRANCH_A_DIR / 'failure_prediction_panel' / 'gradient_boosted_trees_outputs'

HORIZONS = [7, 14, 30]
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
NEG_SUBSAMPLE_PER_POS = 20
DAYS_SINCE_LAST_SENTINEL = 9999.0
SEED = 42
PERMUTATION_SAMPLE_SIZE = 20000
PERMUTATION_REPEATS = 3

PARAM_GRID = [
    {'learning_rate':0.05, 'max_iter':150, 'max_leaf_nodes':15, 'max_depth':None, 'min_samples_leaf':20, 'l2_regularization':0.0},
    {'learning_rate':0.05, 'max_iter':250, 'max_leaf_nodes':31, 'max_depth':None, 'min_samples_leaf':20, 'l2_regularization':0.1},
    {'learning_rate':0.10, 'max_iter':150, 'max_leaf_nodes':31, 'max_depth':None, 'min_samples_leaf':30, 'l2_regularization':1.0},
    {'learning_rate':0.10, 'max_iter':200, 'max_leaf_nodes':63, 'max_depth':None, 'min_samples_leaf':50, 'l2_regularization':1.0},
]

def banner(title):
    print('=' * 100)
    print(title)
    print('=' * 100)

if not PANEL_FILE.is_file():
    raise FileNotFoundError(f'Panel file not found: {PANEL_FILE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

banner('GRADIENT-BOOSTED TREES SETUP')
print('Panel:', PANEL_FILE)
print('Outputs:', OUTPUT_DIR)
print('Horizons:', HORIZONS)
print('Candidate parameter sets:', len(PARAM_GRID))


GRADIENT-BOOSTED TREES SETUP
Panel: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/ecfr_failure_prediction_panel.parquet
Outputs: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/gradient_boosted_trees_outputs
Horizons: [7, 14, 30]
Candidate parameter sets: 4


In [ ]:
# CELL 02 - Load parquet and verify schema
def load_verify(panel_file):
    panel_file = Path(panel_file)
    if not panel_file.is_file():
        raise FileNotFoundError(f'Panel file not found: {panel_file}')
    panel = pd.read_parquet(panel_file)
    if panel.empty:
        raise ValueError('Parquet file is empty.')

    labels = [f'FAILS_WITHIN_{h}D' for h in HORIZONS]
    required = ['ENGINE_SERIAL', 'SNAPSHOT_TIME', 'ENGINE_EVER_FAILS_IN_RECORD'] + labels
    missing = [c for c in required if c not in panel.columns]
    if missing:
        raise ValueError(f'Schema mismatch. Missing required columns: {missing}')
    if panel.ENGINE_SERIAL.isna().any():
        raise ValueError('ENGINE_SERIAL contains missing values.')

    panel['SNAPSHOT_TIME'] = pd.to_datetime(panel.SNAPSHOT_TIME, errors='coerce')
    if panel.SNAPSHOT_TIME.isna().any():
        raise ValueError('SNAPSHOT_TIME contains missing or invalid timestamps.')

    for c in labels + ['ENGINE_EVER_FAILS_IN_RECORD']:
        vals = set(panel[c].dropna().unique().tolist())
        if panel[c].isna().any() or not vals.issubset({0, 1, False, True}):
            raise ValueError(f'{c} must be complete binary 0/1; found {list(vals)[:10]}')
        panel[c] = panel[c].astype(np.int8)

    bad = (panel[labels[0]] > panel[labels[1]]) | (panel[labels[1]] > panel[labels[2]])
    if bad.any():
        raise ValueError(f'{int(bad.sum()):,} rows violate expected 7D <= 14D <= 30D labels.')

    feature_cols = [c for c in panel.columns if c not in set(required)]
    if not feature_cols:
        raise ValueError('No feature columns found.')
    non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(panel[c])]
    if non_numeric:
        raise ValueError(f'Non-numeric feature columns found: {non_numeric[:30]}')

    panel = panel.sort_values(['ENGINE_SERIAL', 'SNAPSHOT_TIME']).reset_index(drop=True)
    schema = pd.DataFrame({
        'column': panel.columns,
        'dtype': panel.dtypes.astype(str).values,
        'missing_count': panel.isna().sum().values,
        'missing_pct': (panel.isna().mean() * 100).round(4).values,
    })

    banner('SCHEMA VERIFIED')
    print(f'Rows: {len(panel):,} | Columns: {panel.shape[1]:,} | Engines: {panel.ENGINE_SERIAL.nunique():,} | Features: {len(feature_cols):,}')
    print('Date range:', panel.SNAPSHOT_TIME.min(), 'to', panel.SNAPSHOT_TIME.max())
    print('Duplicate engine/timestamp rows:', int(panel.duplicated(['ENGINE_SERIAL', 'SNAPSHOT_TIME']).sum()))
    print('Missing feature values before imputation:', int(panel[feature_cols].isna().sum().sum()))
    for c in labels:
        print(f'{c}: positives={int(panel[c].sum()):,} ({panel[c].mean():.6%})')
    display(schema.head(35))
    return panel, labels, feature_cols, schema


In [ ]:
# CELL 03 - Feature preparation and leakage-safe engine split
def prepare_split(panel, labels, features):
    panel = panel.copy()
    days = [c for c in features if c.endswith('__DAYS_SINCE_LAST')]
    other = [c for c in features if c not in days]
    panel[other] = panel[other].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if days:
        panel[days] = panel[days].replace([np.inf, -np.inf], np.nan).fillna(DAYS_SINCE_LAST_SENTINEL)

    X = panel[features].to_numpy(dtype=np.float32)
    Y = panel[labels].to_numpy(dtype=np.int8)
    if not np.isfinite(X).all():
        raise ValueError('Feature matrix contains NaN or infinity after imputation.')

    ever = panel.groupby('ENGINE_SERIAL')['ENGINE_EVER_FAILS_IN_RECORD'].first().astype(int).to_dict()
    engines = np.array(list(ever), dtype=object)
    strat = np.array([ever[e] for e in engines])
    rng = np.random.RandomState(SEED)
    splits = {'train': [], 'val': [], 'test': []}

    for lab in [0, 1]:
        pool = engines[strat == lab].copy()
        rng.shuffle(pool)
        a = int(len(pool) * TRAIN_FRAC)
        b = a + int(len(pool) * VAL_FRAC)
        splits['train'] += list(pool[:a])
        splits['val'] += list(pool[a:b])
        splits['test'] += list(pool[b:])

    splits = {k: set(v) for k, v in splits.items()}
    assert not (splits['train'] & splits['val'])
    assert not (splits['train'] & splits['test'])
    assert not (splits['val'] & splits['test'])

    row_split = np.where(
        panel.ENGINE_SERIAL.isin(splits['train']), 'train',
        np.where(panel.ENGINE_SERIAL.isin(splits['val']), 'val', 'test')
    )

    banner('ENGINE-LEVEL SPLIT')
    for s in ['train', 'val', 'test']:
        idx = np.where(row_split == s)[0]
        print(f'{s:>5}: {len(splits[s]):,} engines | {len(idx):,} rows | ever-fail engines={sum(ever[e] for e in splits[s]):,}')
    print('Feature matrix:', X.shape)
    print('Sentinel-filled DAYS_SINCE_LAST columns:', len(days))
    return panel, X, Y, days, splits, row_split


In [ ]:
# CELL 04 - Training subsample, tuning, metrics, and threshold helpers
def sample_train(y, row_split, seed):
    idx = np.where(row_split == 'train')[0]
    pos = idx[y[idx] == 1]
    neg = idx[y[idx] == 0]
    if len(pos) == 0:
        raise ValueError('No positive training rows for this horizon.')
    rng = np.random.RandomState(seed)
    n = min(len(neg), len(pos) * NEG_SUBSAMPLE_PER_POS)
    keep = rng.choice(neg, size=n, replace=False)
    out = np.concatenate([pos, keep])
    rng.shuffle(out)
    return out

def metrics(y, p):
    if np.unique(y).size < 2:
        return {'roc_auc': np.nan, 'pr_auc': np.nan}
    return {
        'roc_auc': roc_auc_score(y, p),
        'pr_auc': average_precision_score(y, p),
    }

def tune(X, y, tr, va):
    weights = compute_sample_weight(class_weight='balanced', y=y[tr])
    rows = []
    best_model = None
    best_params = None
    best_score = -np.inf

    for params in PARAM_GRID:
        model = HistGradientBoostingClassifier(
            loss='log_loss',
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=SEED,
            **params,
        )
        model.fit(X[tr], y[tr], sample_weight=weights)
        p = model.predict_proba(X[va])[:, 1]
        met = metrics(y[va], p)
        rows.append({**params, 'actual_iterations': model.n_iter_, **met})
        score = -np.inf if np.isnan(met['pr_auc']) else met['pr_auc']
        if score > best_score:
            best_score = score
            best_model = model
            best_params = params.copy()

    if best_model is None:
        raise RuntimeError('No model could be selected from validation results.')
    return best_model, best_params, pd.DataFrame(rows)

def threshold80(y, p):
    if y.sum() == 0:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(y, p)
    candidates = np.where(recall[:-1] >= 0.80)[0]
    if len(candidates) == 0:
        return 0.5
    best = candidates[np.argmax(precision[:-1][candidates])]
    return float(thresholds[best])


In [ ]:
# CELL 05 - Train, evaluate, and save gradient-boosted models
def run_pipeline(panel_file=PANEL_FILE, output_dir=OUTPUT_DIR):
    start = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    panel, labels, features, schema = load_verify(panel_file)
    panel, X, Y, days, splits, row_split = prepare_split(panel, labels, features)
    va = np.where(row_split == 'val')[0]
    te = np.where(row_split == 'test')[0]

    models = {}
    tuning_all = []
    performance = []
    threshold_rows = []
    predictions = pd.DataFrame({
        'ENGINE_SERIAL': panel.loc[te, 'ENGINE_SERIAL'].values,
        'SNAPSHOT_TIME': panel.loc[te, 'SNAPSHOT_TIME'].values,
    })

    for j, horizon in enumerate(HORIZONS):
        y = Y[:, j]
        tr = sample_train(y, row_split, SEED + horizon)
        banner(f'GRADIENT-BOOSTED TREES: {horizon}D')
        print(f'Train subsample={len(tr):,}, positives={int(y[tr].sum()):,}; validation positives={int(y[va].sum()):,}; test positives={int(y[te].sum()):,}')

        model, best_params, tuning = tune(X, y, tr, va)
        tuning.insert(0, 'HORIZON_DAYS', horizon)
        tuning_all.append(tuning)
        models[horizon] = model

        pv = model.predict_proba(X[va])[:, 1]
        pt = model.predict_proba(X[te])[:, 1]
        mv = metrics(y[va], pv)
        mt = metrics(y[te], pt)
        threshold = threshold80(y[va], pv)
        cm05 = confusion_matrix(y[te], pt >= 0.5, labels=[0, 1])
        cmt = confusion_matrix(y[te], pt >= threshold, labels=[0, 1])

        print('Best parameters:', best_params)
        print(f'Validation ROC-AUC={mv["roc_auc"]:.4f}, PR-AUC={mv["pr_auc"]:.4f}')
        print(f'Test ROC-AUC={mt["roc_auc"]:.4f}, PR-AUC={mt["pr_auc"]:.4f}, threshold={threshold:.6f}')

        performance.append({
            'HORIZON_DAYS': horizon,
            'VAL_ROC_AUC': mv['roc_auc'],
            'VAL_PR_AUC': mv['pr_auc'],
            'TEST_ROC_AUC': mt['roc_auc'],
            'TEST_PR_AUC': mt['pr_auc'],
            **{f'BEST_{k.upper()}': v for k, v in best_params.items()},
            'ACTUAL_ITERATIONS': model.n_iter_,
        })
        threshold_rows.append({
            'HORIZON_DAYS': horizon,
            'VALIDATION_THRESHOLD_80_RECALL': threshold,
            'TEST_CM_AT_0.5': cm05.tolist(),
            'TEST_CM_AT_TUNED': cmt.tolist(),
        })
        predictions[f'LABEL_{horizon}D'] = y[te]
        predictions[f'PRED_PROB_{horizon}D'] = pt
        predictions[f'PRED_CLASS_{horizon}D_TUNED'] = (pt >= threshold).astype(int)

        joblib.dump(model, output_dir / f'gradient_boosted_trees_{horizon}d.joblib')

        rng = np.random.RandomState(SEED + horizon)
        imp_idx = rng.choice(va, size=min(PERMUTATION_SAMPLE_SIZE, len(va)), replace=False)
        if np.unique(y[imp_idx]).size == 2:
            importance = permutation_importance(
                model,
                X[imp_idx],
                y[imp_idx],
                scoring='average_precision',
                n_repeats=PERMUTATION_REPEATS,
                random_state=SEED,
                n_jobs=-1,
            )
            pd.DataFrame({
                'feature': features,
                'importance_mean': importance.importances_mean,
                'importance_std': importance.importances_std,
            }).sort_values('importance_mean', ascending=False).to_csv(
                output_dir / f'permutation_importance_{horizon}d.csv', index=False
            )
        else:
            print(f'Permutation importance skipped for {horizon}D because sampled validation rows have one class.')

    performance_df = pd.DataFrame(performance)
    thresholds_df = pd.DataFrame(threshold_rows)
    tuning_df = pd.concat(tuning_all, ignore_index=True)

    schema.to_csv(output_dir / 'verified_schema.csv', index=False)
    performance_df.to_csv(output_dir / 'test_performance_summary.csv', index=False)
    thresholds_df.to_csv(output_dir / 'threshold_sensitivity.csv', index=False)
    tuning_df.to_csv(output_dir / 'hyperparameter_tuning.csv', index=False)
    predictions.to_csv(output_dir / 'test_predictions.csv', index=False)
    joblib.dump(
        {'models': models, 'feature_cols': features, 'horizons': HORIZONS},
        output_dir / 'gradient_boosted_trees_bundle.joblib',
    )

    config = {
        'panel_file': str(panel_file),
        'output_dir': str(output_dir),
        'feature_cols': features,
        'days_since_cols': days,
        'horizons': HORIZONS,
        'seed': SEED,
        'negative_subsample_per_positive': NEG_SUBSAMPLE_PER_POS,
        'permutation_sample_size': PERMUTATION_SAMPLE_SIZE,
        'permutation_repeats': PERMUTATION_REPEATS,
    }
    (output_dir / 'model_config.json').write_text(json.dumps(config, indent=2, default=str))

    banner('PIPELINE COMPLETE')
    display(performance_df)
    print('Artifacts:', output_dir)
    for p in sorted(output_dir.iterdir()):
        print(' -', p.name)
    print(f'Elapsed: {time.time() - start:.1f}s')

    return {
        'models': models,
        'performance': performance_df,
        'thresholds': thresholds_df,
        'predictions': predictions,
        'tuning': tuning_df,
        'output_dir': output_dir,
        'splits': splits,
    }


## Run the complete pipeline

The production input and output paths are defined in Cell 01. Run all cells in order.


In [ ]:
# CELL 06 - Run
results = run_pipeline(PANEL_FILE, OUTPUT_DIR)


SCHEMA VERIFIED
Rows: 1,277,405 | Columns: 35 | Engines: 1,621 | Features: 29
Date range: 1969-02-09 20:16:54 to 2068-05-05 16:57:12
Duplicate engine/timestamp rows: 0
Missing feature values before imputation: 7214127
FAILS_WITHIN_7D: positives=188 (0.014717%)
FAILS_WITHIN_14D: positives=334 (0.026147%)
FAILS_WITHIN_30D: positives=609 (0.047675%)


,column,dtype,missing_count,missing_pct
0,ENGINE_SERIAL,object,0,0.0000
1,SNAPSHOT_TIME,datetime64[ns],0,0.0000
2,chip__COUNT_LOOKBACK,int64,0,0.0000
3,chip__RATE_PER_DAY,float64,0,0.0000
4,chip__DAYS_SINCE_LAST,float64,1238841,96.9811
5,clm_performance__COUNT_LOOKBACK,int64,0,0.0000
6,clm_performance__RATE_PER_DAY,float64,0,0.0000
7,clm_performance__DAYS_SINCE_LAST,float64,744325,58.2685
8,cru_performance__COUNT_LOOKBACK,int64,0,0.0000
9,cru_performance__RATE_PER_DAY,float64,0,0.0000


ENGINE-LEVEL SPLIT
train: 1,134 engines | 906,113 rows | ever-fail engines=131
  val: 242 engines | 185,851 rows | ever-fail engines=28
 test: 245 engines | 185,441 rows | ever-fail engines=29
Feature matrix: (1277405, 29)
Sentinel-filled DAYS_SINCE_LAST columns: 9
GRADIENT-BOOSTED TREES: 7D
Train subsample=2,751, positives=131; validation positives=28; test positives=29
Best parameters: {'learning_rate': 0.1, 'max_iter': 200, 'max_leaf_nodes': 63, 'max_depth': None, 'min_samples_leaf': 50, 'l2_regularization': 1.0}
Validation ROC-AUC=0.9525, PR-AUC=0.1041
Test ROC-AUC=0.9735, PR-AUC=0.1351, threshold=0.901463
GRADIENT-BOOSTED TREES: 14D
Train subsample=4,977, positives=237; validation positives=46; test positives=51
Best parameters: {'learning_rate': 0.1, 'max_iter': 200, 'max_leaf_nodes': 63, 'max_depth': None, 'min_samples_leaf': 50, 'l2_regularization': 1.0}
Validation ROC-AUC=0.9533, PR-AUC=0.1784
Test ROC-AUC=0.9699, PR-AUC=0.1680, threshold=0.876365
GRADIENT-BOOSTED TREES: 30D
T

,HORIZON_DAYS,VAL_ROC_AUC,VAL_PR_AUC,TEST_ROC_AUC,TEST_PR_AUC,BEST_LEARNING_RATE,BEST_MAX_ITER,BEST_MAX_LEAF_NODES,BEST_MAX_DEPTH,BEST_MIN_SAMPLES_LEAF,BEST_L2_REGULARIZATION,ACTUAL_ITERATIONS
0,7,0.952474,0.104136,0.973528,0.135108,0.10,200,63,None,50,1.0,41
1,14,0.953315,0.178448,0.969875,0.167953,0.10,200,63,None,50,1.0,57
2,30,0.924293,0.159336,0.933956,0.170898,0.05,150,15,None,20,0.0,124


Artifacts: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/gradient_boosted_trees_outputs
 - gradient_boosted_trees_14d.joblib
 - gradient_boosted_trees_30d.joblib
 - gradient_boosted_trees_7d.joblib
 - gradient_boosted_trees_bundle.joblib
 - hyperparameter_tuning.csv
 - model_config.json
 - permutation_importance_14d.csv
 - permutation_importance_30d.csv
 - permutation_importance_7d.csv
 - test_performance_summary.csv
 - test_predictions.csv
 - threshold_sensitivity.csv
 - verified_schema.csv
Elapsed: 12.9s
